In [ ]:
import evaluate
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForTokenClassification, TrainingArguments, Trainer, DataCollatorForTokenClassification

## 数据预处理

In [ ]:
ds = load_dataset("lansinuote/peoples-daily-ner")
label_list = ['O','B-PER','I-PER','B-ORG','I-ORG','B-LOC','I-LOC']

In [ ]:
model_name = "bert-base-chinese"
tokenizer = AutoTokenizer.from_pretrained(model_name,local_files_only = True)

#### 注意对齐 token 与 label 

In [ ]:
d = tokenizer('interesting word')
d.word_ids()

In [ ]:
def process_fn(dataset):
    tokenized_set = tokenizer(dataset["tokens"],is_split_into_words=True, max_length = 128, truncation = True, )
    labels = []
    for i, label in enumerate(dataset["ner_tags"]):
        word_ids = tokenized_set.word_ids(batch_index = i)
        sub_label = []
        for word_id in word_ids:
            if word_id is None:
                sub_label.append(-100)
            else: 
                sub_label.append(label[word_id])
        labels.append(sub_label)
    tokenized_set['labels'] = labels
    return tokenized_set    
        

In [ ]:
tokenized_dataset = ds.map(process_fn,batched=True)

## 加载模型与评估函数

In [ ]:
model = AutoModelForTokenClassification.from_pretrained(model_name,local_files_only=True,num_labels = len(label_list))

In [ ]:
model.config.num_labels

In [ ]:
seqeval = evaluate.load('seqeval')
seqeval

In [ ]:
import numpy as np
def eval_metric(pred):
    predictions, labels = pred
    predictions = np.argmax(predictions,axis=-1)
    
    true_predictions = [
        [label_list[p] for p, l in zip(prediciton,label) if l !=-100]
        for prediciton,label in zip(predictions,labels)
    ]
    true_labels = [
        [label_list[l] for p, l in zip(prediciton,label) if l !=-100]
        for prediciton,label in zip(predictions,labels)
    ]
    result = seqeval.compute(predictions=true_predictions,references=true_labels,mode='strict',scheme="IOB2")
    return {
        "f1": result["overall_f1"]
    }

## 设置训练参数进行训练

In [ ]:
args = TrainingArguments(
    output_dir="models_for_ner",
    per_device_train_batch_size=64,
    per_device_eval_batch_size=128,
    eval_strategy="epoch",
    save_strategy="epoch",
    metric_for_best_model="f1",
    load_best_model_at_end=True,
    logging_steps=50,
    num_train_epochs=1
)

In [ ]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    compute_metrics=eval_metric,
    data_collator=DataCollatorForTokenClassification(tokenizer=tokenizer)
)

In [ ]:
trainer.train()

## 评估与预测

In [ ]:
trainer.evaluate(eval_dataset=tokenized_dataset['test'])

In [ ]:
from transformers import pipeline

#### 要指定id2label

In [45]:
model.config.id2label = {idx:label for idx, label in enumerate(label_list)}
model.config

BertConfig {
  "add_cross_attention": false,
  "architectures": [
    "BertForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": null,
  "classifier_dropout": null,
  "directionality": "bidi",
  "dtype": "float32",
  "eos_token_id": null,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "id2label": {
    "0": "O",
    "1": "B-PER",
    "2": "I-PER",
    "3": "B-ORG",
    "4": "I-ORG",
    "5": "B-LOC",
    "6": "I-LOC"
  },
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "is_decoder": false,
  "label2id": {
    "LABEL_0": 0,
    "LABEL_1": 1,
    "LABEL_2": 2,
    "LABEL_3": 3,
    "LABEL_4": 4,
    "LABEL_5": 5,
    "LABEL_6": 6
  },
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 0,
  "pooler_fc_size": 768,
  "pooler_num_attention_heads": 12,
  "pooler_num_fc_layers": 3,
  "pooler_size_per_head": 128,
  "pooler_ty

In [ ]:
ner_pipeline = pipeline("token-classification",model=model,tokenizer = tokenizer,aggregation_strategy = "simple")

In [49]:
res = ner_pipeline("小明在北京上班")
res

[]